# RQ2 source attribution on the full corpus (Colab)

Runs `scripts/rq2_full.py`: Samer's source-attribution protocol on all 7 datasets. That covers 16,682 originals on 2,996 articles, the ChatGPT / PaLM / Dipper / Pegasus rewrites T1-T3, 3 text variants, 4 models and 5×5 article-grouped CV per dataset. This is too long for the laptop (~2-2.5 h with 8 vCPUs).

It is **CPU-only**, so no GPU is needed. Pick **Runtime → Change runtime type → CPU, High-RAM**. An A100 runtime is faster because it has more vCPUs, but it costs more compute units.

**Before running**
1. `MyDrive/sot_colab/sot_colab_bundle.zip` is already on Drive from the semantic run; it provides the corpus and the sample file.
2. On the Mac, build the RQ2 code bundle (from `Sam/`) and upload it to **`MyDrive/sot_colab/`**:
   ```bash
   cd Sam && zip -r -9 data/colab/sot_rq2_bundle.zip src scripts data/derived/sot_style_vocab.json -x '*__pycache__*'
   ```
3. Run all cells.

**Stages** (each saves to Drive and resumes):
1. texts and shared-opening lengths (~5 min);
2. stylometric spaCy features for the 3 variants (~1-1.5 h);
3. cross-validation (~1 h).

**Disconnects:** reconnect and run all cells again. Finished feature chunks and CV jobs are skipped.

**Output:** download `MyDrive/sot_colab/derived/rq2_full_predictions.parquet` (~35 MB) and `rq2_full_texts_meta.parquet` into `Sam/data/derived/`, then run §6 of `notebooks/rq2_source_attribution.ipynb`.

In [ ]:
# 1. Drive, unpack (corpus from the semantic bundle, code + style vocabulary from the RQ2 bundle)
import os, shutil, subprocess, sys, time
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive/sot_colab")
DERIVED, WORK = DRIVE / "derived", Path("/content/work")
for z in ("sot_colab_bundle.zip", "sot_rq2_bundle.zip"):
    assert (DRIVE / z).exists(), f"upload {z} to {DRIVE} first"
print(f"vCPUs: {os.cpu_count()} | RAM: {os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30:.0f} GB")

if not (WORK / "data/sot/paraphrased_datasets").exists():
    shutil.unpack_archive(DRIVE / "sot_colab_bundle.zip", WORK)
shutil.unpack_archive(DRIVE / "sot_rq2_bundle.zip", WORK)   # always: newest code overrides the old bundle's
DERIVED.mkdir(parents=True, exist_ok=True)
for name in ("sot_sample.parquet", "sot_style_vocab.json"):
    if not (DERIVED / name).exists():
        shutil.copy2(WORK / "data/derived" / name, DERIVED / name)
subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"], check=True)
ENV = {**os.environ, "PYTHONPATH": str(WORK / "src"), "TOKENIZERS_PARALLELISM": "false", "OMP_NUM_THREADS": "1",
       "OPENBLAS_NUM_THREADS": "1", "MKL_NUM_THREADS": "1"}   # one thread per worker process


def run(*args, log="rq2_full.log"):
    """Run scripts/rq2_full.py against the Drive caches; stream progress lines and keep the full log on Drive."""
    cmd = [sys.executable, "scripts/rq2_full.py", *args, "--derived", str(DERIVED), "--workers", str(os.cpu_count())]
    print(" ".join(cmd[1:]), flush=True)
    t = time.time()
    with open(DERIVED / log, "a") as f, subprocess.Popen(
            cmd, cwd=WORK, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as p:
        for line in p.stdout:
            if any(k in line for k in ("texts:", "chunk", "cross-validation", "done", "Error", "Traceback", "error")) \
                    or ("cv " in line and "tfidf_char" in line):
                print(line, end="", flush=True)
            f.write(line)
    if p.returncode:
        raise RuntimeError(f"rq2_full.py failed (exit {p.returncode}); see {DERIVED / log}")
    print(f"-> finished in {(time.time() - t) / 60:.1f} min\n", flush=True)


print("ready")

In [ ]:
# 2. Texts -> features -> cross-validation (all resumable), then a completeness check
import pandas as pd

run("all")

preds = pd.read_parquet(DERIVED / "rq2_full_predictions.parquet", columns=["dataset", "ablation", "model", "text_id"])
meta = pd.read_parquet(DERIVED / "rq2_full_texts_meta.parquet")
jobs = preds.groupby(["dataset", "ablation", "model"], observed=True).size()
print(f"{len(preds):,} predictions | {len(jobs)} / 84 (dataset, variant, model) jobs | texts: {len(meta):,} | "
      f"originals: {meta[meta.stage == 'T0'].item_id.nunique():,} on {meta.article_id.nunique():,} articles")
assert len(jobs) == 84 and preds.text_id.nunique() == len(meta), "incomplete: run all cells again to resume"
print(f"Done. Download rq2_full_predictions.parquet and rq2_full_texts_meta.parquet from {DERIVED} "
      "into the repo's Sam/data/derived/.")